In [ ]:
%reload_ext autoreload
%autoreload 2
import numpy as np
from os.path import join
%pylab inline
import sys
sys.path.append('/home/did/MERIS/BRDF/Fourth_Reprocessing/')
from geoutils.figures import Figures
sys.path.insert(0, '/home/did/RTC/SMART-G/tools/')
from luts import LUT, MLUT, Idx, merge, read_lut_hdf, read_mlut_hdf, plot_polar, read_mlut

In [ ]:
## Retrieve MERIS data with S3 format from FTP
DO_FTP = False
import os
import calendar
from ftplib import FTP
dis_dir   = 'MER4VAL/MER4RP_Italy_Land-aero/MEGS_36_44/RR/'
loc_dir   = join('/rfs/data/MERIS/4threprocessing/ftp.acri.fr/', dis_dir)
im_dir    = 'ENV_ME_1_RRG____20090507T093708_20090507T094045_________________0217_078_437______ACR_R_NT____.SEN3/'

if DO_FTP:
    ftp = FTP('ftp.acri.fr')
    ftp.login('ftp_meris4rp','meris%123')
    ftp.cwd(join(dis_dir, im_dir)) 
    ftp.retrlines('NLST', datasets.append)
    os.system('mkdir -p ' + join(loc_dir, im_dir))
    for dataset in datasets :
        ftp.retrbinary('RETR ' + dataset, open(join(loc_dir, im_dir, dataset),'wb').write)

In [ ]:
# Read image
filename = join(loc_dir, im_dir, 'xfdumanifest.xml')
from snappy import Product
from snappy import ProductData
from snappy import ProductIO
from snappy import ProductUtils
from snappy import FlagCoding

print("Reading...")
product = ProductIO.readProduct(filename)
width = product.getSceneRasterWidth()
height = product.getSceneRasterHeight()
name = product.getName()
description = product.getDescription()
band_names = product.getBandNames()
tie_names  = product.getTiePointGridNames()

print("Product:     %s, %s" % (name, description))
print("Raster size: %d x %d pixels" % (width, height))
print("Start time:  " + str(product.getStartTime()))
print("End time:    " + str(product.getEndTime()))
print ''
print("Bands:       %s" % (list(band_names)))
print ''
print("Tie:       %s" % (list(tie_names)))
print ''
for b in product.getTiePointGrids():
    print b.getDescription(), ':', b.getUnit()

In [ ]:
from smacg import Smacg

NB    = 15
NB0   = 1
### band settings
# example : all bands of MERIS
# Read SMAC coefficients
dirname = '/home/did/RTC/smacg'
sensor  = 'MERIS'
aer     = 'CONT' # Desert dust aerosol coefficient
bands_coef = [join(dirname,'COEFFS/coef_'+sensor+str(x+NB0)+'_'+aer+'.dat') for x in range(NB)]

#extract data from image
XSIZE = 512
YSIZE = 512
XOFF  = 64
YOFF  = 64

# start with tie points
tetas       = np.zeros((YSIZE, XSIZE), dtype='float32', order='C')
tetav       = np.zeros((YSIZE, XSIZE), dtype='float32', order='C')
phis        = np.zeros((YSIZE, XSIZE), dtype='float32', order='C')
phiv        = np.zeros((YSIZE, XSIZE), dtype='float32', order='C')
uh2o        = np.zeros((YSIZE, XSIZE), dtype='float32', order='C')
uo3         = np.zeros((YSIZE, XSIZE), dtype='float32', order='C')
taup550     = np.ones(( YSIZE, XSIZE), dtype='float32', order='C') * 0.4
pression    = np.zeros((YSIZE, XSIZE), dtype='float32', order='C')
alt         = np.zeros((YSIZE, XSIZE), dtype='float32', order='C')

rad = product.getTiePointGrid('SZA')
rad.readPixels(YOFF, XOFF, XSIZE, YSIZE, tetas)
rad = product.getTiePointGrid('OZA')
rad.readPixels(YOFF, XOFF, XSIZE, YSIZE, tetav)
rad = product.getTiePointGrid('SAA')
rad.readPixels(YOFF, XOFF, XSIZE, YSIZE, phis)
rad = product.getTiePointGrid('OAA')
rad.readPixels(YOFF, XOFF, XSIZE, YSIZE, phiv)
rad = product.getTiePointGrid('sea_level_pressure')
rad.readPixels(YOFF, XOFF, XSIZE, YSIZE, pression)
rad = product.getTiePointGrid('TP_altitude')
rad.readPixels(YOFF, XOFF, XSIZE, YSIZE, alt)
# computation of surface pressure
pression *= np.exp(-alt/8000.)

rad = product.getTiePointGrid('total_columnar_water_vapour')
rad.readPixels(YOFF, XOFF, XSIZE, YSIZE, uh2o)
# conversion from kg.m-2 to g.cm-2
uh2o *= 0.1
rad = product.getTiePointGrid('total_ozone')
rad.readPixels(YOFF, XOFF, XSIZE, YSIZE, uo3)
# conversion from kg.m-2 to cm.atm
uo3  *= 46.698

# read radiometry
buff  = np.zeros((YSIZE, XSIZE), dtype='float32', order='C') + np.NaN
Es    = np.zeros((YSIZE, XSIZE), dtype='float32', order='C') + np.NaN

bands_es  = ['solar_flux_band_{:d}'.format(x+NB0) for x in range(NB)]
for iband,band in enumerate(bands_es):
    rad = product.getBand(band)
    rad.readPixels(YOFF, XOFF, XSIZE, YSIZE, buff)

bands     = ['M'+'{00:02d}'.format(x+NB0)+'_radiance'     for x in range(NB)]
bands_err = ['M'+'{00:02d}'.format(x+NB0)+'_radiance_err' for x in range(NB)]

rtoa      = np.zeros((NB, YSIZE, XSIZE), dtype='float32', order='C')
rtoa_err  = np.zeros((NB, YSIZE, XSIZE), dtype='float32', order='C')

for iband,(band, band_err, band_es) in enumerate(zip(bands, bands_err, bands_es)):
    rad = product.getBand(band_es)
    rad.readPixels(YOFF, XOFF, XSIZE, YSIZE, Es)
    rad = product.getBand(band)
    rad.readPixels(YOFF, XOFF, XSIZE, YSIZE, buff)
    rtoa[iband,:,:] = buff[:,:]*np.pi/np.cos(tetas*np.pi/180.)/Es[:,:]
    rad = product.getBand(band_err)
    rad.readPixels(YOFF, XOFF, XSIZE, YSIZE, buff)
    rtoa_err[iband,:,:] = buff[:,:]*np.pi/np.cos(tetas*np.pi/180.)/Es[:,:]
    
# compile Smacg and prepare run
S=Smacg()
### GPU grid
XBLOCK  = 256
XGRID   = 256
YGRID   = 1
YBLOCK  = 1

### Data segmentation
M       = XBLOCK * XGRID     # elemnentary size of pixel's array : match the GPU grid
Z       = XSIZE  * YSIZE / M # additional 3rd dimension data : each pixel process 
NLOOP   = 1

# Inputs arrays reshaping
rtoa_in    = np.reshape(rtoa,(NB,Z,XBLOCK,XGRID), order='C')

In [ ]:
%%time
# Run
(rsurf,Jrtoa,Juo3,Juh2o,Jpre,Jtaup) = S.run(bands_coef, tetas, tetav, phis, phiv,
            uh2o, uo3, taup550, pression, rtoa_in,
            XBLOCK=XBLOCK, XGRID=XGRID, NBLOOP=1)

In [ ]:
#fig = Figures(cols=2, rows=1, size=(6,6), fontsize=8)
#fig.imshow(uh2o, title='uh2o',  shrink=0.6, colorbar=True)
#fig.imshow(uo3,  title='uo3',  shrink=0.6, colorbar=True)

In [ ]:
fig1 = Figures(cols=7, rows=NB, size=(4,4), fontsize=8)

for k in range(NB):
    
    fig1.imshow(rtoa[k,:,:], vmin= 0, vmax=0.5, title='rtoa band{00:02d}'.format(k+1),  shrink=0.6, colorbar=True)

    rsurf=rsurf.reshape((NB, XSIZE, YSIZE),order='C')
    fig1.imshow(rsurf[k,:,:],vmin= 0, vmax=0.5, title='rsurf',  shrink=0.6, colorbar=True)

    Jrtoa=Jrtoa.reshape((NB, XSIZE, YSIZE),order='C')
    fig1.imshow(Jrtoa[k,:,:],vmin= 0, vmax=5, title='Jrtoa' ,  shrink=0.6, colorbar=True)

    Juo3=Juo3.reshape((NB, XSIZE, YSIZE),order='C')
    fig1.imshow(Juo3[k,:,:], vmin= -.03, vmax=.03, title='Juo3',  shrink=0.6, colorbar=True)

    Juh2o=Juh2o.reshape((NB, XSIZE, YSIZE),order='C')
    fig1.imshow(Juh2o[k,:,:], vmin= -.03, vmax=.03, title='Juh2o',  shrink=0.6, colorbar=True)

    Jpre=Jpre.reshape((NB, XSIZE, YSIZE),order='C')
    fig1.imshow(Jpre[k,:,:]*1000, vmin= -.3, vmax=.1, title='Jpressure *1e3',  shrink=0.6, colorbar=True)

    Jtaup=Jtaup.reshape((NB, XSIZE, YSIZE),order='C')
    fig1.imshow(Jtaup[k,:,:], vmin= 0, vmax=1, title='Jtaup550',  shrink=0.6, colorbar=True)

In [ ]:
from sympy import *
r,ra,t,tts,ttv,r,s,tg = symbols('r ra t tts ttv r s tg')
diff((r-(ra * t))/ ( t * tts * ttv + (r - (ra * t)) * s),ra)    

In [ ]:
from sympy import *
r,ra,t,tts,ttv,r,s, to3 = symbols('r ra t tts ttv r s to3')
diff((r-(ra * tg*to3))/ ( tg*to3 * tts * ttv + (r - (ra * tg*to3)) * s),to3)    

In [ ]:
diff((r-(ra * t))/ ( t * tts * ttv + (r - (ra * t)) * s),r) 

In [ ]:
from sympy import *
r,ra,t,tts,ttv,r,s, ca_ao3, uo3, ca_no3, m = symbols('r ra t tts ttv r s ca_ao3 uo3 ca_no3 m')
diff (exp ( (ca_ao3)  * pow ( (uo3 *m)  , (ca_no3)  ) ), uo3)

In [ ]:
from sympy import *
rtoa, atm_ref, tg, ttetas, ttetav, s, ttt = symbols('rtoa atm_ref tg ttetas ttetav s ttt')
ttt = tg*ttetas*ttetav
rsurf = (rtoa- atm_ref*tg) / (ttt + (rtoa-atm_ref*tg)*s )
diff (rsurf, tg)

In [ ]:
diff (rsurf, rtoa)

In [ ]:
from sympy import *
ao3,m,no3,uo3,to3 = symbols('ao3 m no3 uo3 to3')
to3 = exp(ao3 * (m*uo3)**no3)

In [ ]:
diff(to3, uo3)

In [ ]:
from sympy import *
rtoa, atm_ref, tgp, to3, ttetas, ttetav, s, ttt = symbols('rtoa atm_ref tgp to3 ttetas ttetav s ttt')
ttt = tgp*to3*ttetas*ttetav
rsurf = (rtoa- atm_ref*tgp*to3) / (ttt + (rtoa-atm_ref*tgp*to3)*s )
diff (rsurf, to3)